# Workbench study template

Run inside an authorized All of Us workspace. Place a clinically reviewed `study.yaml` and a `runtime.json` next to this notebook. See `docs/workbench.md` for reference discovery. This template does not include a real phenotype or credentials.

In [ ]:
from pathlib import Path
import json, os
from aou_studies.specs import StudySpec
from aou_studies.context import discover_context
from aou_studies.source import BigQuerySource
from aou_studies.runner import StudyRun
runtime = json.loads(Path('runtime.json').read_text())
study = StudySpec.load('study.yaml')
if runtime.get('r_library'): os.environ['AOU_R_LIBRARY'] = runtime['r_library']
context = discover_context(dataset=None if runtime.get('resource') else study.dataset, resource=runtime.get('resource'),
                           billing_project=runtime.get('billing_project'), tier=study.tier)
source = BigQuerySource(context, maximum_bytes_billed=runtime['maximum_bytes_billed'])
source.prepare(study)
print(context.summary())

## Dry run

In [ ]:
sql, params = source.extraction_query(study)
source.query(sql, params, dry_run=True)

## Extract, build and match

In [ ]:
assert runtime.get('protocol_confirmed'), 'Confirm the frozen scientific protocol before clinical extraction.'
run = StudyRun(study, runtime['output_directory'])
print(run.extract(source))
print(run.build_groups())
print(run.match())

## Analyze after protocol and balance review

In [ ]:
assert runtime.get('protocol_confirmed'), 'Confirm the protocol before fitting.'
print(run.analyze())
print(run.report())

Review real diagnostics and candidate tables in the authorized workspace. No participant previews or automatic downloads are included.